***SHERIALENS PROJECT***

In [10]:
# Demonstration of building a precision retrieval pipeline (dummy)
# eKLR: Kenya Law Repository - official publisher of Laws of Kenya

def fetch_dummy_law(url):
    # This is a safe placeholder for actual web scraping
    print(f"Fetching legal data from {url} ...")
    return {"title": "Penal Code Sample", "content": "Section 42: land disputes detection."}

# Example usage
law_sample = fetch_dummy_law("https://www.eklr.go.ke/")
print("Sample Law Title:", law_sample["title"])
print("Sample Law Content:", law_sample["content"])

Fetching legal data from https://www.eklr.go.ke/ ...
Sample Law Title: Penal Code Sample
Sample Law Content: Section 42: land disputes detection.


In [11]:
# 📄 Constitution Parser (Simulated Land & Employment Focus)
import pdfplumber
import os
import re
import json

class ConstitutionParser:
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
        self.final_chunks = []

    def process_document(self, start_page=12, end_page=17):
        """Process pages and extract text, tables, images."""
        if not os.path.exists(self.pdf_path):
            print("PDF file not found! Using dummy content for demo.")
            # Dummy content for notebook demo
            self.final_chunks = [
                {"page_number": i, 
                 "text": f"Dummy Article {i}: Land & Employment law excerpt here.",
                 "tables": None,
                 "images": None} 
                for i in range(start_page+1, end_page+1)
            ]
            return self.final_chunks

        with pdfplumber.open(self.pdf_path) as pdf:
            pages_to_process = pdf.pages[start_page:end_page]
            for i, page in enumerate(pages_to_process):
                cropped_page = page.crop(bbox=(0, page.height * 0.17, page.width, page.height * 0.90))
                text = cropped_page.extract_text(x_tolerance=2, y_tolerance=3)
                tables = cropped_page.extract_tables()
                images = cropped_page.images

                page_data = {
                    "page_number": i + start_page + 1,
                    "text": text,
                    "tables": tables,
                    "images": images
                }
                self.final_chunks.append(page_data)
        return self.final_chunks

    def display_sample(self, n=3):
        """Display first n chunks with highlights for Land & Employment."""
        print(f"Displaying first {n} pages of Constitution (Land & Employment focus):\n")
        for chunk in self.final_chunks[:n]:
            text = chunk['text'] or ""
            # Highlight keywords
            for keyword in ["land", "employment", "tenant", "rights"]:
                text = re.sub(f"(?i)({keyword})", r"[\1]", text)
            print(f"Page {chunk['page_number']}:\n{text[:250]}...\n")  # Show first 250 chars for brevity

# Demo Usage
if __name__ == "__main__":
    parser = ConstitutionParser(pdf_path=r"../Datasets/Raw_data/constitution/TheConstitutionOfKenya.pdf")
    parser.process_document()
    parser.display_sample()

PDF file not found! Using dummy content for demo.
Displaying first 3 pages of Constitution (Land & Employment focus):

Page 13:
Dummy Article 13: [Land] & [Employment] law excerpt here....

Page 14:
Dummy Article 14: [Land] & [Employment] law excerpt here....

Page 15:
Dummy Article 15: [Land] & [Employment] law excerpt here....



In [12]:
import google.generativeai as genai

genai.configure(api_key="PASTE_YOUR_API_KEY_HERE")

print("API configured successfully")

API configured successfully


In [13]:


import google.generativeai as genai
import os
import json
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY", "YOUR_API_KEY"))

def get_gemini_embedding(text):

    result = genai.embed_content(
        model="models/text-embedding-004",
        content=text,
        task_type="retrieval_document",
        title="Legal Document Chunk"
    )
    return result['embedding']


In [14]:
def simplify_legal_clause(clause, language="English"):
    """
    Translates complex legalese into plain language.
    Significant contribution: Focuses on accessibility for the 'mwananchi'.
    """
    model = genai.GenerativeModel('gemini-1.5-flash')
    prompt = f"""
    You are SheriaLens AI. Simplify this legal clause for a non-lawyer in {language}.
    Focus on: What does this mean for me? What are my rights?
    
    Clause: {clause}
    """
    response = model.generate_content(prompt)
    return response.text

In [15]:
# -----------------------------
# SHERIA LENS MINI LEGAL AI PIPELINE
# -----------------------------

import google.generativeai as genai

# Configure Gemini API
genai.configure(api_key="YOUR_API_KEY")


# -----------------------------
# 1️⃣ Chunk Legal Text
# -----------------------------
def chunk_legal_text(text, chunk_size=50):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


# -----------------------------
# 2️⃣ Generate Embeddings
# -----------------------------
def get_gemini_embedding(text):

    result = genai.embed_content(
        model="models/embedding-001",
        content=text
    )

    return result["embedding"]


# -----------------------------
# 3️⃣ Simplify Legal Clause
# -----------------------------
def simplify_legal_clause(text, language="English"):

    model = genai.GenerativeModel("gemini-1.5-flash")

    prompt = f"""
    Simplify the following legal clause in simple {language}.
    Make it easy for ordinary citizens to understand.

    Clause:
    {text}
    """

    response = model.generate_content(prompt)

    return response.text



In [17]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# Sample Legal Text (Terms and Conditions snippet)
legal_text = """
1. ACCEPTANCE OF TERMS
By accessing and using this service, you accept and agree to be bound by the terms and provision of this agreement. 
In addition, when using these particular services, you shall be subject to any posted guidelines or rules applicable to such services. 
Any participation in this service will constitute acceptance of this agreement. If you do not agree to abide by the above, please do not use this service.

2. PRIVACY POLICY
The user's privacy is very important to us. Our Privacy Policy is designed to provide you with information about how we collect and use your personal information. 
We encourage you to read the Privacy Policy and to use it to help you make informed decisions.

3. LIMITATION OF LIABILITY
The service and its components are offered for informational purposes only; the service shall not be responsible or liable for the accuracy, 
usefulness or availability of any information transmitted or made available via the service, and shall not be responsible or liable for any error or omissions in that information.
"""

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

cleaned_text = clean_text(legal_text)
print("Cleaned Text Sample:", cleaned_text[:100])

Cleaned Text Sample: 
 acceptance of terms
by accessing and using this service you accept and agree to be bound by the te


In [18]:
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform([cleaned_text])
feature_names = vectorizer.get_feature_names_out()
scores = tfidf_matrix.toarray().flatten()

# Get top 10 keywords
top_keywords = sorted(zip(feature_names, scores), key=lambda x: x[1], reverse=True)[:10]
print("Top Keywords in Document:")
for word, score in top_keywords:
    print(f"{word}: {score:.4f}")

Top Keywords in Document:
service: 0.4657
information: 0.3105
privacy: 0.3105
policy: 0.2328
shall: 0.2328
use: 0.2328
acceptance: 0.1552
agree: 0.1552
agreement: 0.1552
liable: 0.1552


In [19]:
def simple_summarize(text, num_sentences=2):
    sentences = text.split('.')
    # Simple scoring based on keyword presence
    # In a real scenario, we would use word frequency maps
    # For this demo, we just pick the first few for structure
    return '. '.join(sentences[:num_sentences]) + '.'

summary = simple_summarize(legal_text)
print("Simple Summary:")
print(summary)

Simple Summary:

1.  ACCEPTANCE OF TERMS
By accessing and using this service, you accept and agree to be bound by the terms and provision of this agreement.
